In [ ]:
# ===============================
# Dependencies
# ===============================

import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas
import numpy

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

In [ ]:
# ===============================
# Reproducibility
# ===============================

def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed = 50
set_seed(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
# ===============================
# Mount Drive
# ===============================

from google.colab import drive
drive.mount('/content/drive')

DATASET = "/content/drive/MyDrive/dataset.csv"

NUM_API_CALLS = 307
SEQ_LEN = 100

In [ ]:
# ===============================
# Load Dataset
# ===============================

df = pandas.read_csv(DATASET)

train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["malware"],
    random_state=42
)


In [ ]:
# ===============================
# Dataset
# ===============================

class MalwareGraphDataset(Dataset):

    def __init__(self, df):
        self.sequences = df.drop(columns=['hash','malware']).values
        self.labels = torch.tensor(df['malware'].values, dtype=torch.long)

    def __getitem__(self, idx):

        seq = torch.tensor(self.sequences[idx], dtype=torch.long)

        return seq, self.labels[idx]

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = MalwareGraphDataset(train_df)
test_dataset = MalwareGraphDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# ===============================
# Graph Convolution Layer
# ===============================

class GraphConvLayer(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(in_features, out_features) * 0.01
        )

    def forward(self, adj, X):

        B,N,_ = adj.size()

        I = torch.eye(N, device=adj.device).unsqueeze(0)

        A_hat = adj + I

        D = torch.sum(A_hat, dim=2)

        D_inv = torch.diag_embed(1.0/(D+1e-6))

        A_norm = D_inv @ A_hat

        Z = A_norm @ X
        Z = Z @ self.weight

        return Z

In [ ]:
# ===============================
# DGCNN Discriminator
# ===============================

class DGCNN_Discriminator(nn.Module):

    def __init__(self, out_channels=31):

        super().__init__()

        self.gcn = GraphConvLayer(SEQ_LEN, out_channels)

        self.dropout = nn.Dropout(0.6)

        self.fc = nn.Linear(NUM_API_CALLS * out_channels, 3)

    def forward(self, adj, X, return_features=False):

        Z = self.gcn(adj, X)

        Z = F.relu(Z)

        Z = self.dropout(Z)

        features = Z.reshape(Z.size(0), -1)

        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits

In [ ]:
# ===============================
# Generator
# ===============================

LATENT_DIM = 128
EMB_DIM = 128

class Generator(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(NUM_API_CALLS, EMB_DIM)

        self.init_fc = nn.Linear(LATENT_DIM, 256)

        self.rnn = nn.GRU(
            input_size=EMB_DIM,
            hidden_size=256,
            batch_first=True
        )

        self.token_proj = nn.Linear(256, NUM_API_CALLS)

        self.start_token = nn.Parameter(torch.zeros(1,1,EMB_DIM))

    def forward(self, z):

        batch_size = z.size(0)

        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)

        inputs = self.start_token.repeat(batch_size, SEQ_LEN,1)

        outputs,_ = self.rnn(inputs,h0)

        logits = self.token_proj(outputs)

        probs = F.gumbel_softmax(logits, tau=0.5, hard=True)

        tokens = torch.argmax(probs, dim=-1)

        return tokens

In [ ]:
# ===============================
# Models
# ===============================

D = DGCNN_Discriminator().to(DEVICE)
G = Generator().to(DEVICE)

opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))


In [ ]:
# ===============================
# Helper
# ===============================

def seq_to_graph(seq_batch):

    B = seq_batch.size(0)

    src = seq_batch[:, :-1]
    dst = seq_batch[:, 1:]

    adj = torch.zeros(
        (B, NUM_API_CALLS, NUM_API_CALLS),
        device=seq_batch.device
    )

    batch_index = torch.arange(B, device=seq_batch.device).unsqueeze(1)

    adj[batch_index, src, dst] = 1

    X = F.one_hot(seq_batch, NUM_API_CALLS).float().permute(0,2,1)

    return adj, X

In [ ]:
# ===============================
# Training
# ===============================

EPOCHS = 50

history = {
    "epoch": [],
    "d_loss": [],
    "g_loss": [],
}

for epoch in range(EPOCHS):

    for seq_real, labels in train_loader:
        seq_real = seq_real.to(DEVICE)
        labels = labels.to(DEVICE)

        adj_real, X_real = seq_to_graph(seq_real)

        adj_real = adj_real.to(DEVICE)
        X_real = X_real.to(DEVICE)
        labels = labels.to(DEVICE)

        B = labels.size(0)

        # ------------------
        # Train Discriminator
        # ------------------

        opt_D.zero_grad()

        logits_real = D(adj_real, X_real)

        loss_real = F.cross_entropy(
            logits_real,
            labels
        )

        z = torch.randn(B, LATENT_DIM).to(DEVICE)

        fake_seq = G(z)

        adj_fake, X_fake = seq_to_graph(fake_seq)

        logits_fake = D(adj_fake, X_fake)

        fake_labels = torch.full(
            (B,),
            2,
            device=DEVICE
        )

        loss_fake = F.cross_entropy(
            logits_fake,
            fake_labels
        )

        loss_D = loss_real + loss_fake

        loss_D.backward()
        opt_D.step()

        # ------------------
        # Train Generator
        # ------------------

        opt_G.zero_grad()

        z = torch.randn(B, LATENT_DIM).to(DEVICE)

        fake_seq = G(z)

        adj_fake, X_fake = seq_to_graph(fake_seq)

        logits_fake, feat_fake = D(adj_fake, X_fake, True)

        _, feat_real = D(adj_real, X_real, True)

        loss_G = F.mse_loss(
            feat_fake.mean(0),
            feat_real.mean(0)
        )

        loss_G.backward()
        opt_G.step()

    print(f"Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}")

    history["epoch"].append(epoch + 1)
    history["d_loss"].append(loss_D.item())
    history["g_loss"].append(loss_G.item())


In [ ]:
history_df = pandas.DataFrame(history)

history_df.to_csv(
    f"dgcnn_sgan_history_seed_{seed}.csv",
    index=False
)

In [ ]:
torch.save(
    D.state_dict(),
    f"dgvgan_seed_{seed}.pt"
)

In [ ]:
D.eval()

all_probs = []
all_labels = []

with torch.no_grad():

    for seq, labels in test_loader:

        seq = seq.to(DEVICE)
        labels = labels.to(DEVICE)

        adj, X = seq_to_graph(seq)

        logits = D(adj, X)

        probs = F.softmax(logits[:, :2], dim=1)

        all_probs.extend(probs[:,1].cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

preds = [1 if x > 0.5 else 0 for x in all_probs]

print(classification_report(all_labels, preds))

print("PR-AUC:", average_precision_score(all_labels, all_probs))

print("ROC-AUC:", roc_auc_score(all_labels, all_probs))